<a href="https://colab.research.google.com/github/Karthikreddy1010/BeadSegCount-Deep-Learning-Pipeline-for-Microbead-Segmentation-and-Quantification/blob/main/Macroplasticbeads_seg_counting_April.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/Microplastic.zip
!unzip /content/GT_Counts.zip

In [3]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras import mixed_precision
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error
from scipy import stats
from scipy.optimize import linear_sum_assignment
import pandas as pd
from skimage.feature import peak_local_max
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# GPU SETUP
# ============================================================================

gpus = tf.config.list_physical_devices('GPU')
if len(gpus) == 0:
    print("⚠️  No GPU detected, running on CPU")
else:
    print("✅ GPU detected:", gpus)
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

mixed_precision.set_global_policy('mixed_float16')
print("✅ Mixed precision enabled")

np.random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# CONFIGURATION
# ============================================================================

IMG_HEIGHT = 512
IMG_WIDTH  = 512
BATCH_SIZE = 32
EPOCHS     = 100
PATIENCE   = 15
LEARNING_RATE = 1e-4

# Adaptive sigma: sigma = SIGMA_K * sqrt(component_area)
SIGMA_K = 0.25

# Boundary dilation radius (pixels)
BOUNDARY_DILATION = 3

# Primary counting method: boundary-guided peak detection
CENTER_PEAK_THRESHOLD  = 0.5
# UPGRADE 3 — min_distance is now adaptive (derived from sigma); this is fallback
CENTER_MIN_DISTANCE_FALLBACK = 6
# Scale factor k for: min_distance = mean_sigma * MIN_DIST_SIGMA_K
MIN_DIST_SIGMA_K = 1.5

# Boundary-guided peak suppression threshold
BOUNDARY_SUPPRESSION_THRESHOLD = 0.3

# UPGRADE 1 — GT-calibrated soft-count normalization
# Set to None → computed from training data; set to float to override
SOFT_COUNT_NORM = None

# UPGRADE 4 — Center sharpness loss weight
LAMBDA_SHARPNESS = 0.1

# Ablation flags (all True = full model)
USE_ATTENTION_GATES      = True
USE_SCSE                 = True
USE_BOUNDARY_HEAD        = True
USE_CENTER_HEAD          = True
USE_AUGMENTATION         = True

# Loss weights
LAMBDA_SEG            = 1.0
LAMBDA_CENTER         = 1.0
LAMBDA_BOUNDARY       = 0.5
LAMBDA_CB_CONSISTENCY = 0.2

# UPGRADE 5 — Instance-aware evaluation
# Distance threshold (pixels) for matching predicted center to GT center
INSTANCE_MATCH_DIST = 8

# Paths
TRAIN_IMAGE_DIR = '/content/Microplastic/Training/Original'
TRAIN_MASK_DIR  = 'Microplastic/Training/Masks'
VAL_IMAGE_DIR   = 'Microplastic/Valid/Original'
VAL_MASK_DIR    = 'Microplastic/Valid/masks'
TEST_IMAGE_DIR  = 'Microplastic/testing/Original'
TEST_MASK_DIR   = 'Microplastic/testing/Masks'

TRAIN_COUNT_CSV = 'Gt_train.csv'
VAL_COUNT_CSV   = 'Gt_valid.csv'
TEST_COUNT_CSV  = 'Gt_test.csv'

# ============================================================================
# 1. DATA AUGMENTATION
# ============================================================================

def apply_augmentations(image, mask, center_map, boundary_map, prob=0.5):
    """Consistent augmentations across all heads."""
    if not USE_AUGMENTATION or np.random.random() > prob:
        return image, mask, center_map, boundary_map

    if np.random.random() > 0.5:
        image        = np.fliplr(image).copy()
        mask         = np.fliplr(mask).copy()
        center_map   = np.fliplr(center_map).copy()
        boundary_map = np.fliplr(boundary_map).copy()

    if np.random.random() > 0.5:
        image        = np.flipud(image).copy()
        mask         = np.flipud(mask).copy()
        center_map   = np.flipud(center_map).copy()
        boundary_map = np.flipud(boundary_map).copy()

    k = np.random.randint(0, 4)
    if k > 0:
        image        = np.rot90(image,        k=k).copy()
        mask         = np.rot90(mask,         k=k).copy()
        center_map   = np.rot90(center_map,   k=k).copy()
        boundary_map = np.rot90(boundary_map, k=k).copy()

    if np.random.random() > 0.7:
        alpha = np.random.uniform(0.9, 1.1)
        beta  = np.random.uniform(-0.05, 0.05)
        image = np.clip(alpha * image + beta, 0, 1)

    return image, mask, center_map, boundary_map

# ============================================================================
# 2. PREPROCESSING
# ============================================================================

def preprocess_image(image):
    """CLAHE enhancement for microplastic visibility."""
    if image.dtype != np.uint8:
        image_uint8 = (image * 255).astype(np.uint8)
    else:
        image_uint8 = image.copy()

    lab     = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    lab_out = cv2.merge((l_clahe, a, b))
    rgb_out = cv2.cvtColor(lab_out, cv2.COLOR_LAB2RGB)
    return rgb_out.astype(np.float32) / 255.0


def preprocess_mask(mask):
    """Binarize mask and remove small noise."""
    binary = (mask > 127).astype(np.float32)
    if binary.sum() > 0:
        kernel = np.ones((2, 2), np.uint8)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    return binary

# ============================================================================
# 3. GROUND TRUTH GENERATION
# ============================================================================

def generate_center_heatmap(binary_mask, sigma_k=SIGMA_K):
    """
    Generate a Gaussian center heatmap with adaptive per-instance sigma.

        sigma_i = sigma_k * sqrt(area_i)

    Returns
    -------
    heatmap      : np.ndarray (H, W) float32 in [0, 1]
    centers      : list of (row, col, sigma) tuples
    mean_sigma   : float — mean sigma across all instances (used for
                   adaptive min_distance at inference, UPGRADE 3)
    """
    H, W    = binary_mask.shape[:2]
    heatmap = np.zeros((H, W), dtype=np.float32)
    centers = []
    sigmas  = []

    binary_u8  = (binary_mask > 0.5).astype(np.uint8)
    num_labels, labels = cv2.connectedComponents(binary_u8, connectivity=8)

    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)

    for label_id in range(1, num_labels):
        component = labels == label_id
        area      = int(component.sum())
        if area < 3:
            continue

        sigma = max(sigma_k * np.sqrt(float(area)), 1.5)
        sigmas.append(sigma)

        ys, xs = np.where(component)
        cy = float(ys.mean())
        cx = float(xs.mean())
        centers.append((cy, cx, sigma))

        gauss   = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma ** 2))
        heatmap = np.maximum(heatmap, gauss)

    mean_sigma = float(np.mean(sigmas)) if sigmas else float(CENTER_MIN_DISTANCE_FALLBACK)
    return heatmap, centers, mean_sigma


def generate_boundary_map(binary_mask, dilation_radius=BOUNDARY_DILATION):
    """Generate a dilated boundary map from a binary mask."""
    binary_u8 = (binary_mask > 0.5).astype(np.uint8) * 255
    edges     = cv2.Canny(binary_u8, 50, 150)

    if dilation_radius > 0:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * dilation_radius + 1, 2 * dilation_radius + 1)
        )
        edges = cv2.dilate(edges, kernel, iterations=1)

    return (edges > 0).astype(np.float32)

# ============================================================================
# 4. UPGRADE 1 — GT-CALIBRATED SOFT-COUNT NORMALIZATION
# ============================================================================

def calibrate_soft_count_norm(center_maps_gt, gt_counts):
    """
    Compute the normalization factor for soft counting from ground-truth data.

        norm_factor = mean( sum(center_gt_i) / gt_count_i )   for all i with count > 0

    This is the data-driven calibration that replaces the earlier heuristic
    estimation.  Call this ONCE on training data and fix the result.

    Parameters
    ----------
    center_maps_gt : np.ndarray (N, H, W, 1)  — GT center heatmaps
    gt_counts      : np.ndarray (N,)           — GT particle counts

    Returns
    -------
    norm_factor : float
    """
    ratios = []
    for cm, count in zip(center_maps_gt, gt_counts):
        if count > 0:
            total = float(np.squeeze(cm).sum())
            ratios.append(total / float(count))

    if not ratios:
        print("  ⚠️  No valid samples for soft-count calibration; using fallback=25.0")
        return 25.0

    norm_factor = float(np.mean(ratios))
    print(f"  ✅ Calibrated soft-count norm_factor = {norm_factor:.4f}  "
          f"(median={np.median(ratios):.4f}, std={np.std(ratios):.4f})")
    return norm_factor

# ============================================================================
# 5. UPGRADE 3 — ADAPTIVE MIN_DISTANCE
# ============================================================================

def adaptive_min_distance(center_map, sigma_k=MIN_DIST_SIGMA_K,
                           fallback=CENTER_MIN_DISTANCE_FALLBACK):
    """
    Estimate adaptive min_distance from the predicted center heatmap.

        σ_eff = sqrt( sum_of_peaks_area / (n_peaks_est * π) )
        min_distance = max(σ_eff * sigma_k, fallback)

    Because σ is proportional to sqrt(area), the inter-peak spacing must
    be at least proportional to σ to avoid merging adjacent particles.

    Parameters
    ----------
    center_map : np.ndarray (H, W) or (H, W, 1)
    sigma_k    : float — multiplier (default 1.5, ~1.5 sigma separation)
    fallback   : int   — minimum value if estimation fails

    Returns
    -------
    min_dist : int
    """
    cm    = np.squeeze(center_map).astype(np.float32)
    above = (cm > CENTER_PEAK_THRESHOLD).sum()
    if above == 0:
        return fallback

    # Rough peak count for sigma estimation
    n_peaks_est = max(1, int(cm.max() * 5))
    sigma_eff   = np.sqrt(float(above) / (n_peaks_est * np.pi))
    min_dist    = int(max(sigma_eff * sigma_k, fallback))
    return min_dist

# ============================================================================
# 6. COUNTING: BOUNDARY-GUIDED PEAK DETECTION (PRIMARY) + SOFT COUNT (AUX)
# ============================================================================

def count_from_center_map(center_map,
                          boundary_map=None,
                          threshold=CENTER_PEAK_THRESHOLD,
                          min_distance=None,
                          boundary_threshold=BOUNDARY_SUPPRESSION_THRESHOLD):
    """
    PRIMARY counting method: boundary-guided peak detection with
    adaptive min_distance (UPGRADE 3).

    Parameters
    ----------
    center_map         : np.ndarray (H, W) or (H, W, 1)
    boundary_map       : np.ndarray (H, W) or (H, W, 1), optional
    threshold          : float
    min_distance       : int or None  — if None, computed adaptively
    boundary_threshold : float

    Returns
    -------
    count       : int
    valid_peaks : np.ndarray (N, 2) — (row, col) of valid centers
    """
    cm = np.squeeze(center_map).astype(np.float32)

    if cm.max() == 0:
        return 0, np.empty((0, 2), dtype=int)

    # UPGRADE 3 — adaptive min_distance
    if min_distance is None:
        min_distance = adaptive_min_distance(cm)

    raw_peaks = peak_local_max(
        cm,
        min_distance=min_distance,
        threshold_abs=threshold,
        exclude_border=True
    )

    if boundary_map is None or len(raw_peaks) == 0:
        return len(raw_peaks), raw_peaks

    bm = np.squeeze(boundary_map).astype(np.float32)
    valid_peaks = [
        (r, c) for (r, c) in raw_peaks
        if bm[r, c] < boundary_threshold
    ]

    valid_peaks = (np.array(valid_peaks, dtype=int)
                   if valid_peaks
                   else np.empty((0, 2), dtype=int))
    return len(valid_peaks), valid_peaks


def soft_count(center_map, norm_factor):
    """
    AUXILIARY counting method: integrate the predicted center heatmap.

        count_soft = sum(center_map) / norm_factor

    norm_factor must be GT-calibrated (UPGRADE 1) — do NOT use heuristic
    estimation at inference.

    Parameters
    ----------
    center_map  : np.ndarray (H, W) or (H, W, 1)
    norm_factor : float  — calibrated from training GT (UPGRADE 1)

    Returns
    -------
    count_soft : float
    """
    cm = np.squeeze(center_map).astype(np.float32)
    return float(cm.sum() / max(norm_factor, 1e-6))

# ============================================================================
# 7. UPGRADE 5 — INSTANCE-AWARE EVALUATION (CENTER MATCHING F1)
# ============================================================================

def instance_detection_metrics(gt_centers_list, pred_peaks_list,
                                match_dist=INSTANCE_MATCH_DIST):
    """
    Compute instance-level detection F1, Precision, and Recall by matching
    predicted peaks to GT centers using the Hungarian algorithm.

    A predicted peak is a True Positive if it is within `match_dist` pixels
    of an unmatched GT center.  Unmatched GTs are False Negatives; unmatched
    predictions are False Positives.

    Parameters
    ----------
    gt_centers_list  : list of (N_i, 2 or 3) arrays  — GT (row, col[, sigma])
    pred_peaks_list  : list of (M_i, 2) arrays        — predicted (row, col)
    match_dist       : float — maximum matching distance in pixels

    Returns
    -------
    dict with keys: precision, recall, f1, tp, fp, fn
    """
    total_tp = total_fp = total_fn = 0

    for gt_centers, pred_peaks in zip(gt_centers_list, pred_peaks_list):
        # Use only (row, col), strip sigma if present
        gt_rc   = np.array(gt_centers, dtype=float)[:, :2] if len(gt_centers) else np.empty((0, 2))
        pred_rc = np.array(pred_peaks, dtype=float)

        n_gt   = len(gt_rc)
        n_pred = len(pred_rc)

        if n_gt == 0 and n_pred == 0:
            continue
        elif n_gt == 0:
            total_fp += n_pred
            continue
        elif n_pred == 0:
            total_fn += n_gt
            continue

        # Build distance matrix (n_gt x n_pred)
        diff   = gt_rc[:, None, :] - pred_rc[None, :, :]   # (n_gt, n_pred, 2)
        dist   = np.sqrt((diff ** 2).sum(axis=-1))          # (n_gt, n_pred)

        # Hungarian matching on valid pairs only
        row_idx, col_idx = linear_sum_assignment(dist)

        tp = 0
        matched_gt   = set()
        matched_pred = set()
        for r, c in zip(row_idx, col_idx):
            if dist[r, c] <= match_dist:
                tp += 1
                matched_gt.add(r)
                matched_pred.add(c)

        fp = n_pred - len(matched_pred)
        fn = n_gt   - len(matched_gt)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall    = total_tp / (total_tp + total_fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return {
        'precision': precision,
        'recall'   : recall,
        'f1'       : f1,
        'tp'       : total_tp,
        'fp'       : total_fp,
        'fn'       : total_fn,
    }

# ============================================================================
# 8. ARCHITECTURE COMPONENTS
# ============================================================================

def attention_gate(skip, gating, inter_channels, name="attn_gate"):
    theta = layers.Conv2D(inter_channels, 1, padding='same',
                          kernel_initializer='he_normal',
                          name=f"{name}_theta")(skip)
    phi   = layers.Conv2D(inter_channels, 1, padding='same',
                          kernel_initializer='he_normal',
                          name=f"{name}_phi")(gating)
    f     = layers.Activation('relu', name=f"{name}_add")(
                layers.Add()([theta, phi]))
    psi   = layers.Conv2D(1, 1, padding='same',
                          kernel_initializer='he_normal',
                          name=f"{name}_psi")(f)
    alpha = layers.Activation('sigmoid', name=f"{name}_alpha")(psi)
    return layers.Multiply(name=f"{name}_out")([skip, alpha])


def scse_block(x, reduction=16, name="scse"):
    channels = x.shape[-1]
    gap  = layers.GlobalAveragePooling2D(name=f"{name}_gap")(x)
    fc1  = layers.Dense(channels // reduction, activation='relu',
                        kernel_initializer='he_normal',
                        name=f"{name}_fc1")(gap)
    fc2  = layers.Dense(channels, activation='sigmoid',
                        kernel_initializer='he_normal',
                        name=f"{name}_fc2")(fc1)
    cse  = layers.Reshape((1, 1, channels), name=f"{name}_reshape")(fc2)
    x_c  = layers.Multiply(name=f"{name}_c_mul")([x, cse])
    sse  = layers.Conv2D(1, 1, activation='sigmoid',
                         kernel_initializer='he_normal',
                         name=f"{name}_sse_conv")(x)
    x_s  = layers.Multiply(name=f"{name}_s_mul")([x, sse])
    return layers.Add(name=f"{name}_out")([x_c, x_s])


def residual_block(x, filters, name="res_block"):
    shortcut = x

    x = layers.Conv2D(filters, 3, padding='same',
                      kernel_initializer='he_normal',
                      name=f"{name}_conv1")(x)
    x = layers.BatchNormalization(dtype='float32', name=f"{name}_bn1")(x)
    x = layers.Activation('relu', name=f"{name}_relu1")(x)

    x = layers.Conv2D(filters, 3, padding='same',
                      kernel_initializer='he_normal',
                      name=f"{name}_conv2")(x)
    x = layers.BatchNormalization(dtype='float32', name=f"{name}_bn2")(x)

    gap  = layers.GlobalAveragePooling2D(name=f"{name}_gap")(x)
    fc1  = layers.Dense(max(filters // 8, 1), activation='relu',
                        kernel_initializer='he_normal',
                        name=f"{name}_fc1")(gap)
    fc2  = layers.Dense(filters, activation='sigmoid',
                        kernel_initializer='he_normal',
                        name=f"{name}_fc2")(fc1)
    attn = layers.Reshape((1, 1, filters), name=f"{name}_reshape")(fc2)
    x    = layers.Multiply(name=f"{name}_attn_mul")([x, attn])

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same',
                                 kernel_initializer='he_normal',
                                 name=f"{name}_shortcut_conv")(shortcut)
        shortcut = layers.BatchNormalization(dtype='float32',
                                            name=f"{name}_shortcut_bn")(shortcut)

    x = layers.Add(name=f"{name}_add")([shortcut, x])
    x = layers.Activation('relu', name=f"{name}_out")(x)
    return x

# ============================================================================
# 9. CENTER–BOUNDARY DUAL-SUPERVISION U-NET
# ============================================================================

def build_center_boundary_unet(input_shape=(256, 256, 3)):
    inputs = layers.Input(shape=input_shape)

    # ── ENCODER ──────────────────────────────────────────────────
    c1 = layers.Conv2D(64, 3, padding='same', kernel_initializer='he_normal')(inputs)
    c1 = layers.BatchNormalization(dtype='float32')(c1)
    c1 = layers.Activation('relu')(c1)
    c1 = layers.Conv2D(64, 3, padding='same', kernel_initializer='he_normal')(c1)
    c1 = layers.BatchNormalization(dtype='float32')(c1)
    c1 = layers.Activation('relu')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = residual_block(p1, 128,  name="enc2")
    p2 = layers.MaxPooling2D()(c2)

    c3 = residual_block(p2, 256,  name="enc3")
    p3 = layers.MaxPooling2D()(c3)

    c4 = residual_block(p3, 512,  name="enc4")
    p4 = layers.MaxPooling2D()(c4)

    # ── BOTTLENECK ────────────────────────────────────────────────
    c5 = residual_block(p4, 1024, name="bottleneck")

    # ── SHARED DECODER ───────────────────────────────────────────
    def decoder_block(up_input, skip, skip_channels, out_channels,
                      gate_name, scse_name, block_name):
        u = layers.UpSampling2D()(up_input)
        if USE_ATTENTION_GATES:
            skip_gated = attention_gate(skip, u, skip_channels // 2,
                                        name=gate_name)
        else:
            skip_gated = skip
        u = layers.concatenate([u, skip_gated])
        u = layers.Conv2D(out_channels, 3, padding='same',
                          kernel_initializer='he_normal',
                          name=f"{block_name}_conv1")(u)
        u = layers.BatchNormalization(dtype='float32',
                                      name=f"{block_name}_bn1")(u)
        u = layers.Activation('relu', name=f"{block_name}_relu1")(u)
        u = layers.Conv2D(out_channels, 3, padding='same',
                          kernel_initializer='he_normal',
                          name=f"{block_name}_conv2")(u)
        u = layers.BatchNormalization(dtype='float32',
                                      name=f"{block_name}_bn2")(u)
        if USE_SCSE:
            u = scse_block(u, name=scse_name)
        u = layers.Activation('relu', name=f"{block_name}_relu2")(u)
        return u

    d6 = decoder_block(c5, c4, 512, 512,
                       gate_name="gate_c4", scse_name="scse_d6",
                       block_name="dec6")
    d7 = decoder_block(d6, c3, 256, 256,
                       gate_name="gate_c3", scse_name="scse_d7",
                       block_name="dec7")
    d8 = decoder_block(d7, c2, 128, 128,
                       gate_name="gate_c2", scse_name="scse_d8",
                       block_name="dec8")

    u9 = layers.UpSampling2D()(d8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(64, 3, padding='same', kernel_initializer='he_normal',
                       name="dec9_conv1")(u9)
    c9 = layers.BatchNormalization(dtype='float32', name="dec9_bn1")(c9)
    c9 = layers.Activation('relu', name="dec9_relu1")(c9)
    c9 = layers.Conv2D(64, 3, padding='same', kernel_initializer='he_normal',
                       name="dec9_conv2")(c9)
    c9 = layers.BatchNormalization(dtype='float32', name="dec9_bn2")(c9)
    c9 = layers.Activation('relu', name="dec9_relu2")(c9)

    # ── OUTPUT HEADS ─────────────────────────────────────────────
    seg_output = layers.Conv2D(
        1, 1, activation='sigmoid',
        name='seg_output', dtype='float32'
    )(c9)

    center_output = layers.Conv2D(
        1, 1, activation='sigmoid',
        name='center_output', dtype='float32'
    )(c9)

    boundary_output = layers.Conv2D(
        1, 1, activation='sigmoid',
        name='boundary_output', dtype='float32'
    )(c9)

    return tf.keras.Model(
        inputs, [seg_output, center_output, boundary_output],
        name="CenterBoundary_UNet_v3"
    )

# ============================================================================
# 10. LOSS FUNCTIONS
# ============================================================================

def dice_coefficient(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    inter  = tf.reduce_sum(y_true * y_pred)
    union  = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return (2. * inter + smooth) / (union + smooth)


def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75):
    y_true  = tf.cast(y_true, tf.float32)
    y_pred  = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1 - 1e-7)
    tp      = tf.reduce_sum(y_true * y_pred)
    fp      = tf.reduce_sum((1 - y_true) * y_pred)
    fn      = tf.reduce_sum(y_true * (1 - y_pred))
    tversky = (tp + 1e-7) / (tp + alpha * fp + beta * fn + 1e-7)
    return tf.pow(1 - tversky, gamma)


def segmentation_loss(y_true, y_pred):
    return focal_tversky_loss(y_true, y_pred)


def focal_bce(y_true, y_pred, gamma=2.0, pos_weight=10.0):
    y_true  = tf.cast(y_true, tf.float32)
    y_pred  = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1 - 1e-7)
    bce_pos = -y_true       * tf.math.log(y_pred)
    bce_neg = -(1 - y_true) * tf.math.log(1 - y_pred)
    pt      = y_true * y_pred + (1 - y_true) * (1 - y_pred)
    focal   = tf.pow(1.0 - pt, gamma)
    return tf.reduce_mean(focal * (pos_weight * bce_pos + bce_neg))


def center_heatmap_loss(y_true, y_pred):
    """MSE + Focal BCE for center heatmap."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    mse    = tf.reduce_mean(tf.square(y_true - y_pred))
    f_bce  = focal_bce(y_true, y_pred, gamma=2.0, pos_weight=10.0)
    return mse + 0.1 * f_bce


def boundary_loss(y_true, y_pred):
    y_true  = tf.cast(y_true, tf.float32)
    y_pred  = tf.cast(y_pred, tf.float32)
    d_loss  = 1 - dice_coefficient(y_true, y_pred)
    bce     = focal_bce(y_true, y_pred, gamma=2.0, pos_weight=5.0)
    return d_loss + 0.5 * bce


def cb_consistency_loss(center_pred, boundary_pred):
    """Penalise predicted centers that overlap predicted boundaries."""
    center_pred   = tf.cast(center_pred,   tf.float32)
    boundary_pred = tf.cast(boundary_pred, tf.float32)
    return tf.reduce_mean(center_pred * boundary_pred)


# ── UPGRADE 4 — Center sharpness loss ────────────────────────────────────────

def center_sharpness_loss(center_pred, pool_size=5):
    """
    Encourage sharp, localised peaks in the center heatmap.

        L_sharp = mean( (center_pred - max_pool(center_pred))^2 )

    A perfect Dirac-like peak would have centre_pred == max_pool(centre_pred)
    everywhere; a blurry prediction will have many non-peak pixels where
    centre_pred < max_pool and contribute heavily to this loss.

    Parameters
    ----------
    center_pred : tf.Tensor (B, H, W, 1)
    pool_size   : int — neighbourhood size for max-pooling (must be odd)

    Returns
    -------
    loss : scalar tf.Tensor
    """
    center_f32 = tf.cast(center_pred, tf.float32)
    # Max-pool with same padding preserves spatial size
    max_pooled  = tf.nn.max_pool2d(
        center_f32,
        ksize=pool_size,
        strides=1,
        padding='SAME'
    )
    return tf.reduce_mean(tf.square(center_f32 - max_pooled))

# ============================================================================
# 11. CUSTOM TRAINING MODEL
# ============================================================================

class CenterBoundaryModel(tf.keras.Model):
    """
    Wraps the U-Net to inject:
      • Center–boundary consistency loss  (UPGRADE 4 v2)
      • Center sharpness loss             (UPGRADE 4 — new)
    into every training step.
    """

    def __init__(self, unet,
                 lambda_cb=LAMBDA_CB_CONSISTENCY,
                 lambda_sharp=LAMBDA_SHARPNESS,
                 **kwargs):
        super().__init__(**kwargs)
        self.unet         = unet
        self.lambda_cb    = lambda_cb
        self.lambda_sharp = lambda_sharp

    def call(self, inputs, training=False):
        return self.unet(inputs, training=training)

    def train_step(self, data):
        x, y_dict = data

        y_seg      = y_dict['seg_output']
        y_center   = y_dict['center_output']
        y_boundary = y_dict['boundary_output']

        with tf.GradientTape() as tape:
            seg_pred, center_pred, boundary_pred = self(x, training=True)

            l_seg      = segmentation_loss(y_seg,      seg_pred)
            l_center   = center_heatmap_loss(y_center,  center_pred)
            l_boundary = boundary_loss(y_boundary,      boundary_pred)
            l_cb       = cb_consistency_loss(center_pred, boundary_pred)
            # UPGRADE 4
            l_sharp    = center_sharpness_loss(center_pred)

            total_loss = (LAMBDA_SEG         * l_seg
                          + LAMBDA_CENTER    * l_center
                          + LAMBDA_BOUNDARY  * l_boundary
                          + self.lambda_cb   * l_cb
                          + self.lambda_sharp * l_sharp)

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.compiled_metrics.update_state(
            [y_seg, y_center, y_boundary],
            [seg_pred, center_pred, boundary_pred]
        )

        results = {m.name: m.result() for m in self.metrics}
        results.update({
            'loss'      : total_loss,
            'l_seg'     : l_seg,
            'l_center'  : l_center,
            'l_boundary': l_boundary,
            'l_cb'      : l_cb,
            'l_sharp'   : l_sharp,
        })
        return results

    def test_step(self, data):
        x, y_dict = data

        y_seg      = y_dict['seg_output']
        y_center   = y_dict['center_output']
        y_boundary = y_dict['boundary_output']

        seg_pred, center_pred, boundary_pred = self(x, training=False)

        l_seg      = segmentation_loss(y_seg,      seg_pred)
        l_center   = center_heatmap_loss(y_center,  center_pred)
        l_boundary = boundary_loss(y_boundary,      boundary_pred)
        l_cb       = cb_consistency_loss(center_pred, boundary_pred)
        l_sharp    = center_sharpness_loss(center_pred)

        total_loss = (LAMBDA_SEG         * l_seg
                      + LAMBDA_CENTER    * l_center
                      + LAMBDA_BOUNDARY  * l_boundary
                      + self.lambda_cb   * l_cb
                      + self.lambda_sharp * l_sharp)

        self.compiled_metrics.update_state(
            [y_seg, y_center, y_boundary],
            [seg_pred, center_pred, boundary_pred]
        )

        results = {m.name: m.result() for m in self.metrics}
        results.update({'val_loss': total_loss, 'val_l_cb': l_cb,
                        'val_l_sharp': l_sharp})
        return results

# ============================================================================
# 12. DATA LOADING
# ============================================================================

def normalize_filename(name):
    name = os.path.basename(name)
    name = os.path.splitext(name)[0]
    name = name.replace('_mask', '')
    return name


def load_dataset(image_dir, mask_dir, csv_path=None, aug=False):
    """Load images, masks, center heatmaps (adaptive sigma), boundary maps.
    Also returns per-sample mean_sigma for UPGRADE 3 (adaptive min_distance).
    """
    print(f"Loading data from: {image_dir}")

    image_files = sorted([
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])

    count_map = {}
    if csv_path and os.path.exists(csv_path):
        try:
            df = pd.read_csv(csv_path)
            if {'filename', 'gt_particle_count'}.issubset(df.columns):
                for _, row in df.iterrows():
                    count_map[normalize_filename(row['filename'])] = \
                        float(row['gt_particle_count'])
                print(f"  Loaded {len(count_map)} GT counts from CSV")
        except Exception as e:
            print(f"  Warning: Could not load CSV: {e}")

    images        = []
    masks         = []
    center_maps   = []
    boundary_maps = []
    counts        = []
    gt_centers_all = []   # list of (cy, cx, sigma) for each image
    mean_sigmas   = []

    for img_path in image_files:
        try:
            fname     = os.path.basename(img_path)
            fname_key = normalize_filename(fname)

            mask_path = None
            for ext in ['_mask.png', '_mask.jpg', '.png', '.jpg',
                        '_mask.PNG', '_mask.JPG']:
                candidate = os.path.join(mask_dir, fname_key + ext)
                if os.path.exists(candidate):
                    mask_path = candidate
                    break
            if not mask_path:
                continue

            img     = Image.open(img_path).convert('RGB')
            img     = img.resize((IMG_WIDTH, IMG_HEIGHT))
            img_arr = preprocess_image(np.array(img, dtype=np.float32) / 255.0)

            msk     = Image.open(mask_path).convert('L')
            msk     = msk.resize((IMG_WIDTH, IMG_HEIGHT))
            msk_arr = preprocess_mask(np.array(msk, dtype=np.float32))

            cmap, centers, mean_sigma = generate_center_heatmap(msk_arr, sigma_k=SIGMA_K)
            bmap                      = generate_boundary_map(msk_arr, dilation_radius=BOUNDARY_DILATION)

            if fname_key in count_map:
                particle_count = count_map[fname_key]
            else:
                particle_count = float(len(centers))

            msk_arr_ch = msk_arr[..., np.newaxis]
            cmap_ch    = cmap[..., np.newaxis]
            bmap_ch    = bmap[..., np.newaxis]

            if aug:
                img_arr, msk_arr_ch, cmap_ch, bmap_ch = apply_augmentations(
                    img_arr, msk_arr_ch, cmap_ch, bmap_ch, prob=0.5
                )

            images.append(img_arr)
            masks.append(msk_arr_ch)
            center_maps.append(cmap_ch)
            boundary_maps.append(bmap_ch)
            counts.append(particle_count)
            gt_centers_all.append(centers)
            mean_sigmas.append(mean_sigma)

        except Exception as e:
            print(f"  Error loading {img_path}: {e}")
            continue

    if not images:
        raise ValueError(f"No valid images loaded from {image_dir}")

    print(f"  Loaded {len(images)} samples")
    return (
        np.array(images,        dtype=np.float32),
        np.array(masks,         dtype=np.float32),
        np.array(center_maps,   dtype=np.float32),
        np.array(boundary_maps, dtype=np.float32),
        np.array(counts,        dtype=np.float32),
        gt_centers_all,   # list of lists (not a uniform array)
        np.array(mean_sigmas, dtype=np.float32),
    )

# ============================================================================
# 13. MAIN PIPELINE
# ============================================================================

def main():
    print("=" * 80)
    print("CENTER–BOUNDARY DUAL-SUPERVISION U-NET v3 — MICROPLASTIC COUNTING")
    print("Upgrades: GT-Calibrated Soft Count | Adaptive min_distance | "
          "Sharpness Loss | Instance F1")
    print("=" * 80)

    print(f"\nConfiguration:")
    print(f"  Attention Gates        : {USE_ATTENTION_GATES}")
    print(f"  scSE                   : {USE_SCSE}")
    print(f"  Center head            : {USE_CENTER_HEAD}")
    print(f"  Boundary head          : {USE_BOUNDARY_HEAD}")
    print(f"  Augmentation           : {USE_AUGMENTATION}")
    print(f"  Adaptive sigma_k       : {SIGMA_K}")
    print(f"  CB-Consistency λ       : {LAMBDA_CB_CONSISTENCY}")
    print(f"  Sharpness λ            : {LAMBDA_SHARPNESS}")
    print(f"  λ_seg / λ_ctr / λ_bnd : "
          f"{LAMBDA_SEG} / {LAMBDA_CENTER} / {LAMBDA_BOUNDARY}")

    # ── LOAD DATA ────────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("LOADING DATA")
    print("=" * 80)

    def load_with_fallback(img_dir, msk_dir, csv, aug):
        try:
            return load_dataset(img_dir, msk_dir, csv, aug)
        except Exception as e:
            print(f"  Warning ({e}), retrying without CSV …")
            return load_dataset(img_dir, msk_dir, None, aug)

    (X_train, y_train, C_train, B_train, train_counts,
     train_gt_centers, train_mean_sigmas) = load_with_fallback(
        TRAIN_IMAGE_DIR, TRAIN_MASK_DIR, TRAIN_COUNT_CSV, aug=True)

    (X_val,   y_val,   C_val,   B_val,   val_counts,
     val_gt_centers,   val_mean_sigmas)   = load_with_fallback(
        VAL_IMAGE_DIR,   VAL_MASK_DIR,   VAL_COUNT_CSV,   aug=False)

    (X_test,  y_test,  C_test,  B_test,  test_counts,
     test_gt_centers,  test_mean_sigmas)  = load_with_fallback(
        TEST_IMAGE_DIR,  TEST_MASK_DIR,  TEST_COUNT_CSV,  aug=False)

    print(f"\nDataset sizes — Train: {len(X_train)}, "
          f"Val: {len(X_val)}, Test: {len(X_test)}")

    # ── UPGRADE 1 — GT-calibrated soft-count normalization ───────
    print("\n── GT-Calibrated Soft-Count Normalization (UPGRADE 1) ──")
    norm_factor = calibrate_soft_count_norm(C_train, train_counts)

    # ── BUILD MODEL ──────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("BUILDING MODEL")
    print("=" * 80)

    np.random.seed(42)
    tf.random.set_seed(42)

    unet  = build_center_boundary_unet(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    model = CenterBoundaryModel(unet,
                                lambda_cb=LAMBDA_CB_CONSISTENCY,
                                lambda_sharp=LAMBDA_SHARPNESS)

    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=1e-4,
        clipnorm=1.0
    )

    model.compile(
        optimizer=optimizer,
        metrics={
            'seg_output'      : [dice_coefficient],
            'center_output'   : ['mse'],
            'boundary_output' : ['accuracy'],
        }
    )

    _ = model(X_train[:1])
    model.unet.summary()

    # ── TRAINING ─────────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("TRAINING")
    print("=" * 80)

    train_targets = {
        'seg_output'      : y_train,
        'center_output'   : C_train,
        'boundary_output' : B_train,
    }
    val_targets = {
        'seg_output'      : y_val,
        'center_output'   : C_val,
        'boundary_output' : B_val,
    }

    ckpt_name = (f'best_model_v3_center{USE_CENTER_HEAD}'
                 f'_boundary{USE_BOUNDARY_HEAD}'
                 f'_attn{USE_ATTENTION_GATES}.weights.h5')

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_seg_output_dice_coefficient',
            patience=PATIENCE,
            restore_best_weights=True,
            mode='max', verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_seg_output_dice_coefficient',
            factor=0.5, patience=8, min_lr=1e-6, mode='max', verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            ckpt_name,
            monitor='val_seg_output_dice_coefficient',
            save_best_only=True,
            save_weights_only=True,
            mode='max', verbose=1
        ),
        tf.keras.callbacks.TensorBoard(
            log_dir=f'logs_v3_center{USE_CENTER_HEAD}_boundary{USE_BOUNDARY_HEAD}',
            histogram_freq=0, write_graph=True
        ),
    ]

    history = model.fit(
        X_train, train_targets,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_val, val_targets),
        callbacks=callbacks,
        verbose=1, shuffle=True
    )

    # ── EVALUATION ───────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("EVALUATION")
    print("=" * 80)

    print("Loading best checkpoint weights …")
    model.load_weights(ckpt_name)

    print("Generating test predictions …")
    seg_pred, center_pred, boundary_pred = model.predict(X_test, verbose=1)

    # ── SEGMENTATION METRICS ─────────────────────────────────────
    SEG_THRESHOLD = 0.35
    y_pred_bin    = (seg_pred > SEG_THRESHOLD).astype(np.uint8)
    y_true_flat   = y_test.flatten().astype(np.uint8)
    y_pred_flat   = y_pred_bin.flatten()

    seg_metrics = {
        'dice'     : (2 * np.sum(y_true_flat * y_pred_flat) /
                      (np.sum(y_true_flat) + np.sum(y_pred_flat) + 1e-8)),
        'iou'      : (np.sum(y_true_flat * y_pred_flat) /
                      (np.sum(y_true_flat) + np.sum(y_pred_flat)
                       - np.sum(y_true_flat * y_pred_flat) + 1e-8)),
        'precision': precision_score(y_true_flat, y_pred_flat, zero_division=0),
        'recall'   : recall_score(y_true_flat,    y_pred_flat, zero_division=0),
        'f1'       : f1_score(y_true_flat,         y_pred_flat, zero_division=0),
    }

    print("\n── Segmentation (pixel-level) ────────────────────────")
    for k, v in seg_metrics.items():
        print(f"  {k:<12}: {v:.4f}")

    # ── PRIMARY: Boundary-guided peak detection ───────────────────
    print("\n── Counting — PRIMARY: Boundary-Guided Peak Detection (UPGRADE 3) ──")
    predicted_counts_peak = []
    all_pred_peaks        = []
    gt_counts             = test_counts.flatten()

    for i in range(len(center_pred)):
        # UPGRADE 3 — adaptive min_distance
        cnt_peak, peaks = count_from_center_map(
            center_pred[i],
            boundary_map=boundary_pred[i],
            threshold=CENTER_PEAK_THRESHOLD,
            min_distance=None,                # triggers adaptive computation
            boundary_threshold=BOUNDARY_SUPPRESSION_THRESHOLD
        )
        predicted_counts_peak.append(cnt_peak)
        all_pred_peaks.append(peaks)

    predicted_counts_peak = np.array(predicted_counts_peak)

    # ── AUXILIARY: GT-calibrated soft count ──────────────────────
    print("\n── Counting — AUXILIARY: GT-Calibrated Soft Count (UPGRADE 1) ──")
    predicted_counts_soft = np.array([
        soft_count(center_pred[i], norm_factor=norm_factor)
        for i in range(len(center_pred))
    ])

    def print_count_metrics(label, predicted):
        mae     = mean_absolute_error(gt_counts, predicted)
        rmse    = np.sqrt(mean_squared_error(gt_counts, predicted))
        bias    = float(np.mean(predicted - gt_counts))
        tol_acc = float(np.mean(np.abs(predicted - gt_counts) <= 1) * 100)
        pearson = (np.corrcoef(gt_counts, predicted)[0, 1]
                   if np.std(gt_counts) > 0 and np.std(predicted) > 0
                   else 0.0)
        print(f"\n  [{label}]")
        print(f"    MAE      : {mae:.3f}")
        print(f"    RMSE     : {rmse:.3f}")
        print(f"    Bias     : {bias:+.3f}")
        print(f"    Tol-Acc  : {tol_acc:.1f}%  (±1 particle)")
        print(f"    Pearson r: {pearson:.3f}")
        return mae, rmse, bias, tol_acc, pearson

    m_peak = print_count_metrics("PRIMARY — Boundary-Guided Peak Detection",
                                 predicted_counts_peak)
    m_soft = print_count_metrics("AUXILIARY — GT-Calibrated Soft Count",
                                 predicted_counts_soft)

    # ── UPGRADE 5 — Instance-aware evaluation ────────────────────
    print("\n── Instance-Aware Evaluation (UPGRADE 5) ─────────────")
    inst_metrics = instance_detection_metrics(
        test_gt_centers,
        all_pred_peaks,
        match_dist=INSTANCE_MATCH_DIST
    )
    print(f"  Instance Precision : {inst_metrics['precision']:.4f}")
    print(f"  Instance Recall    : {inst_metrics['recall']:.4f}")
    print(f"  Instance F1        : {inst_metrics['f1']:.4f}")
    print(f"  TP / FP / FN       : "
          f"{inst_metrics['tp']} / {inst_metrics['fp']} / {inst_metrics['fn']}")

    # ── VISUALISATION ─────────────────────────────────────────────
    out_dir = (f'results_v3_center{USE_CENTER_HEAD}'
               f'_boundary{USE_BOUNDARY_HEAD}')
    os.makedirs(out_dir, exist_ok=True)

    def visualise_sample(idx):
        fig, axes = plt.subplots(2, 4, figsize=(18, 9))

        axes[0, 0].imshow(X_test[idx])
        axes[0, 0].set_title('Input Image')

        axes[0, 1].imshow(y_test[idx].squeeze(), cmap='gray')
        axes[0, 1].set_title(f'GT Mask  (count={int(gt_counts[idx])})')

        axes[0, 2].imshow(C_test[idx].squeeze(), cmap='hot')
        axes[0, 2].set_title('GT Center Heatmap (adaptive σ)')

        axes[0, 3].imshow(B_test[idx].squeeze(), cmap='gray')
        axes[0, 3].set_title('GT Boundary Map')

        im = axes[1, 0].imshow(seg_pred[idx].squeeze(), cmap='jet',
                               vmin=0, vmax=1)
        axes[1, 0].set_title('Predicted Segmentation')
        plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

        im2 = axes[1, 1].imshow(center_pred[idx].squeeze(), cmap='hot',
                                vmin=0, vmax=1)
        axes[1, 1].set_title('Predicted Center Heatmap')
        plt.colorbar(im2, ax=axes[1, 1], fraction=0.046, pad=0.04)

        # UPGRADE 3 — adaptive min_distance peaks
        _, valid_peaks = count_from_center_map(
            center_pred[idx],
            boundary_map=boundary_pred[idx],
            threshold=CENTER_PEAK_THRESHOLD,
            min_distance=None,
            boundary_threshold=BOUNDARY_SUPPRESSION_THRESHOLD
        )
        _, raw_peaks = count_from_center_map(
            center_pred[idx],
            boundary_map=None,
            threshold=CENTER_PEAK_THRESHOLD,
            min_distance=None
        )

        axes[1, 2].imshow(center_pred[idx].squeeze(), cmap='hot')
        if len(raw_peaks):
            axes[1, 2].scatter(raw_peaks[:, 1], raw_peaks[:, 0],
                               c='white', s=40, marker='+', linewidths=1,
                               label='raw', alpha=0.5)
        if len(valid_peaks):
            axes[1, 2].scatter(valid_peaks[:, 1], valid_peaks[:, 0],
                               c='cyan', s=50, marker='x', linewidths=1.5,
                               label='boundary-filtered')
        axes[1, 2].legend(fontsize=7, loc='upper right')
        axes[1, 2].set_title(
            f'Peaks: raw={len(raw_peaks)} → filtered={len(valid_peaks)}\n'
            f'min_dist={adaptive_min_distance(center_pred[idx])}'
        )

        axes[1, 3].imshow(boundary_pred[idx].squeeze(), cmap='gray')
        axes[1, 3].set_title('Predicted Boundary')

        for ax in axes.flat:
            ax.axis('off')

        cnt_soft = round(soft_count(center_pred[idx], norm_factor=norm_factor))
        plt.suptitle(
            f'Sample {idx} | GT={int(gt_counts[idx])} | '
            f'Peak(BG)={predicted_counts_peak[idx]} | '
            f'Soft(aux)={cnt_soft}',
            fontsize=12, fontweight='bold'
        )
        plt.tight_layout()
        plt.savefig(f'{out_dir}/sample_{idx}.png', dpi=150, bbox_inches='tight')
        plt.close()

    for i in range(min(5, len(X_test))):
        visualise_sample(i)

    # ── SUMMARY PLOTS ─────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    mx = max(gt_counts.max(),
             predicted_counts_peak.max(),
             predicted_counts_soft.max()) + 1

    # (a) PRIMARY — peak counting scatter
    axes[0].scatter(gt_counts, predicted_counts_peak,
                    alpha=0.65, s=35, edgecolors='k', linewidth=0.5,
                    color='#2196F3', label='Primary: Boundary-Guided Peak')
    axes[0].plot([0, mx], [0, mx], 'r--', alpha=0.7)
    axes[0].set_xlabel('GT Count')
    axes[0].set_ylabel('Predicted Count')
    mae_p, _, _, _, pearson_p = m_peak
    axes[0].set_title(f'PRIMARY: Boundary-Guided Peaks\nMAE={mae_p:.2f}, r={pearson_p:.3f}')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # (b) AUXILIARY — soft counting scatter
    axes[1].scatter(gt_counts, predicted_counts_soft,
                    alpha=0.65, s=35, edgecolors='k', linewidth=0.5,
                    color='#FF9800', label='Auxiliary: Soft Count')
    axes[1].plot([0, mx], [0, mx], 'r--', alpha=0.7)
    axes[1].set_xlabel('GT Count')
    axes[1].set_ylabel('Predicted Count')
    mae_s, _, _, _, pearson_s = m_soft
    axes[1].set_title(f'AUXILIARY: Soft Count (calibrated)\nMAE={mae_s:.2f}, r={pearson_s:.3f}')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    # (c) Instance-aware F1 bar chart (UPGRADE 5)
    inst_labels = ['Precision', 'Recall', 'F1']
    inst_vals   = [inst_metrics['precision'], inst_metrics['recall'],
                   inst_metrics['f1']]
    bars = axes[2].bar(inst_labels, inst_vals, color=['#9C27B0', '#E91E63', '#F44336'],
                       alpha=0.85)
    for bar, val in zip(bars, inst_vals):
        axes[2].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    axes[2].set_ylim(0, 1.15)
    axes[2].set_title(
        f'Instance Detection (UPGRADE 5)\n'
        f'TP={inst_metrics["tp"]} FP={inst_metrics["fp"]} FN={inst_metrics["fn"]}'
    )
    axes[2].grid(True, alpha=0.3, axis='y')

    # (d) Segmentation bar chart
    metrics_labels = ['Dice', 'IoU', 'Precision', 'Recall', 'F1']
    metrics_vals   = [seg_metrics[k.lower()] for k in metrics_labels]
    bars2 = axes[3].bar(metrics_labels, metrics_vals, color='#4CAF50', alpha=0.85)
    for bar, val in zip(bars2, metrics_vals):
        axes[3].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    axes[3].set_ylim(0, 1.15)
    axes[3].set_title('Segmentation Metrics (pixel-level)')
    axes[3].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f'{out_dir}/performance_summary.png', dpi=300, bbox_inches='tight')
    plt.show()

    # ── SAVE RESULTS ─────────────────────────────────────────────
    results_df = pd.DataFrame({
        'GT_Count'                : gt_counts,
        'Pred_Count_Peak_BG'      : predicted_counts_peak,   # PRIMARY
        'Pred_Count_Soft'         : predicted_counts_soft,   # AUXILIARY
        'Abs_Error_Peak'          : np.abs(predicted_counts_peak - gt_counts),
        'Abs_Error_Soft'          : np.abs(predicted_counts_soft - gt_counts),
    })
    results_df.to_csv(f'{out_dir}/counting_results.csv', index=False)

    with open(f'{out_dir}/summary.txt', 'w') as f:
        f.write("CENTER–BOUNDARY DUAL-SUPERVISION U-NET v3 — RESULTS\n")
        f.write("=" * 60 + "\n\n")
        f.write("Upgrades applied:\n")
        f.write("  [1] Adaptive sigma heatmap (σ = SIGMA_K * sqrt(area))\n")
        f.write("  [2] Focal BCE for center & boundary losses\n")
        f.write("  [3] Center-boundary consistency loss during training\n")
        f.write("  [4] Boundary-guided peak filtering at inference\n")
        f.write("  [NEW-1] GT-calibrated soft-count normalization\n")
        f.write(f"          norm_factor = {norm_factor:.4f}\n")
        f.write("  [NEW-2] Adaptive min_distance from sigma (replaces fixed=6)\n")
        f.write("  [NEW-3] Center sharpness loss (L_sharp)\n")
        f.write("  [NEW-4] Instance-aware evaluation (Hungarian F1)\n\n")
        f.write(f"PRIMARY counting method  : Boundary-Guided Peak Detection\n")
        f.write(f"AUXILIARY counting method: GT-Calibrated Soft Count\n\n")
        f.write("Architecture:\n")
        f.write(f"  Attention Gates : {USE_ATTENTION_GATES}\n")
        f.write(f"  scSE            : {USE_SCSE}\n")
        f.write(f"  Center head     : {USE_CENTER_HEAD}\n")
        f.write(f"  Boundary head   : {USE_BOUNDARY_HEAD}\n\n")
        f.write("Segmentation (pixel-level):\n")
        for k, v in seg_metrics.items():
            f.write(f"  {k:<12}: {v:.4f}\n")
        f.write("\nCounting — PRIMARY (Boundary-Guided Peak Detection):\n")
        labels = ['MAE', 'RMSE', 'Bias', 'Tol-Acc', 'Pearson']
        for lbl, val in zip(labels, m_peak):
            f.write(f"  {lbl:<12}: {val:.3f}\n")
        f.write("\nCounting — AUXILIARY (GT-Calibrated Soft Count):\n")
        for lbl, val in zip(labels, m_soft):
            f.write(f"  {lbl:<12}: {val:.3f}\n")
        f.write("\nInstance Detection (UPGRADE 5 — Hungarian F1):\n")
        f.write(f"  Precision    : {inst_metrics['precision']:.4f}\n")
        f.write(f"  Recall       : {inst_metrics['recall']:.4f}\n")
        f.write(f"  F1           : {inst_metrics['f1']:.4f}\n")
        f.write(f"  TP/FP/FN     : {inst_metrics['tp']} / "
                f"{inst_metrics['fp']} / {inst_metrics['fn']}\n")
        f.write("\nAblation study configs:\n")
        f.write("  Config 1 — Baseline (no upgrades)\n")
        f.write("  Config 2 — + Adaptive sigma only\n")
        f.write("  Config 3 — + Focal BCE only\n")
        f.write("  Config 4 — + CB-Consistency only\n")
        f.write("  Config 5 — + Sharpness Loss only\n")
        f.write("  Config 6 — Full model (all upgrades)\n")

    print(f"\n✅ Results written to: {out_dir}/")
    print(f"   sample_*.png | performance_summary.png | "
          f"counting_results.csv | summary.txt")
    print(f"\n  Soft-count norm_factor used: {norm_factor:.4f}")
    print("\n" + "=" * 80)
    print("✅ PIPELINE COMPLETE")
    print("=" * 80)


if __name__ == "__main__":
    main()

✅ GPU detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Mixed precision enabled
CENTER–BOUNDARY DUAL-SUPERVISION U-NET v3 — MICROPLASTIC COUNTING
Upgrades: GT-Calibrated Soft Count | Adaptive min_distance | Sharpness Loss | Instance F1

Configuration:
  Attention Gates        : True
  scSE                   : True
  Center head            : True
  Boundary head          : True
  Augmentation           : True
  Adaptive sigma_k       : 0.25
  CB-Consistency λ       : 0.2
  Sharpness λ            : 0.1
  λ_seg / λ_ctr / λ_bnd : 1.0 / 1.0 / 0.5

LOADING DATA
Loading data from: /content/Microplastic/Training/Original
  Loaded 825 GT counts from CSV
  Loaded 825 samples
Loading data from: Microplastic/Valid/Original
  Loaded 79 GT counts from CSV
  Loaded 79 samples
Loading data from: Microplastic/testing/Original
  Loaded 39 GT counts from CSV
  Loaded 39 samples

Dataset sizes — Train: 825, Val: 79, Test: 39

── GT-Calibrated Soft-Count Normalization (UPGRADE 

Model: "CenterBoundary_UNet_v3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 512, 512,  │      1,792 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512, 512,  │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 512, 512,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 512, 512,  │     36,928 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512, 512,  │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 512, 512,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 256, 256,  │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_conv1 (Conv2D) │ (None, 256, 256,  │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_bn1            │ (None, 256, 256,  │        512 │ enc2_conv1[0][0]  │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_relu1          │ (None, 256, 256,  │          0 │ enc2_bn1[0][0]    │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_conv2 (Conv2D) │ (None, 256, 256,  │    147,584 │ enc2_relu1[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_bn2            │ (None, 256, 256,  │        512 │ enc2_conv2[0][0]  │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_gap            │ (None, 128)       │          0 │ enc2_bn2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_fc1 (Dense)    │ (None, 16)        │      2,064 │ enc2_gap[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_fc2 (Dense)    │ (None, 128)       │      2,176 │ enc2_fc1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc2_shortcut_conv  │ (None, 256, 256,  │      8,320 │ max_pooling2d_4[… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 33,021,169 (125.97 MB)

 Trainable params: 33,005,553 (125.91 MB)

 Non-trainable params: 15,616 (61.00 KB)


TRAINING
Epoch 1/100


TypeError: argument of type 'NoneType' is not iterable